In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
import os, glob, numpy as np, pandas as pd, xarray as xr
O=rf'{DATA_ROOT}/new_climate/_proj_main'; TB=os.path.join(O,'tables')
M2C=rf'{DATA_ROOT}/AIS_processing/panel_cache_M2/grid2'
CUTS={'wave':[1.25,2.5,4.0,6.0,9.0],'wind':[8.0,10.8,13.9,17.2,20.8]}
REG={'NW_Europe':[(36,45,-15,-6),(45,63,-15,-6),(43,63,-6,0),(48,63,0,13)],
 'NW_Atlantic':[(18,31,-98,-60),(31,42,-82,-60),(42,50,-67,-60),(45,55,-60,-46)],
 'Med_Black':[(30,43,-6,0),(30,46,0,19),(30,46,19,26.5),(30,41,26.5,37),(41,47.5,26.5,42)],
 'Asia':[(-10,20,95,130),(20,46,117,146)],
 'Indian_Ocean_N':[(5,30,50,78),(5,26,78,95)]}
cells=xr.open_dataarray(os.path.join(O,'obs_wave.nc')).cell.values
t0=pd.to_datetime(xr.open_dataarray(os.path.join(O,'cm_wave_ACCESS_his.nc')).time.values)
k0=(t0.year>=2002)&(t0.year<=2014); GOOD=np.ones(len(cells),bool)
for m in ('ACCESS','ECEARTH'):
    for kd in ('wave','wind'):
        GOOD&=np.isfinite(xr.open_dataarray(os.path.join(O,f'cm_{kd}_{m}_his.nc')).load().values[k0]).mean(0)>=0.9
def reg_of(g):
    la,lo=map(float,g.split('_'))
    for r,bs in REG.items():
        for S,N,W,E in bs:
            if S<=la<N and W<=lo<E: return r
    return None
RG=np.array([reg_of(g) for g in cells])
pg=pd.read_parquet(M2C+r'\panel_global.parquet',columns=['region','grid_id','E_it'])
pg=pg[pg.grid_id.isin(set(cells[GOOD]))]
WR=pg.groupby('region').E_it.sum(); WR=WR/WR.sum()          # region weights (traffic)

def annual_agg(arr,times,kind):
    """→ DataFrame[year, cut, value] (traffic-weighted mean annual days across regions)"""
    yr=times.year.values; u=np.unique(yr); idx=np.searchsorted(yr,u); ok=np.isfinite(arr)
    out=[]
    for c in CUTS[kind]:
        cnt=np.add.reduceat((ok&(arr>=c)).astype(np.float32),idx,0)
        v=sum(cnt[:,(RG==r)&GOOD].mean(1)*WR[r] for r in REG)
        out.append(pd.DataFrame({'year':u,'cut':c,'value':v}))
    return pd.concat(out,ignore_index=True)

def from_cache(kind,y0,y1):
    fr=[]
    for f in glob.glob(os.path.join(M2C,f'*__dmax_{kind}.parquet')):
        d=pd.read_parquet(f); d['date']=pd.to_datetime(d['date'])
        fr.append(d[(d.date.dt.year>=y0)&(d.date.dt.year<=y1)])
    d=pd.concat(fr,ignore_index=True).drop_duplicates(['grid_id','date'])
    p=d.pivot(index='date',columns='grid_id',values='dmax').sort_index().reindex(columns=cells)
    return p.values.astype(np.float32), pd.DatetimeIndex(p.index)

# Baseline (ERA5 2002-2020 annual mean)
BASE={}
for kind in ('wave','wind'):
    o=xr.open_dataarray(os.path.join(O,f'obs_{kind}.nc')).load()
    a,t=from_cache(kind,2015,2020)
    arr=np.concatenate([o.values,a]); tim=pd.DatetimeIndex(np.concatenate([pd.to_datetime(o.time.values),np.asarray(t)]))
    s=np.argsort(tim.values)
    BASE[kind]=annual_agg(arr[s],tim[s],kind).groupby('cut').value.mean()
    print(f'baseline {kind}:', BASE[kind].round(1).to_dict())

rows=[]
for kind in ('wave','wind'):
    for m in ('ACCESS','ECEARTH'):
        for ssp in ('ssp126','ssp585'):
            q=xr.open_dataarray(os.path.join(O,f'qdm_{kind}_{m}_{ssp}.nc')).load()
            d=annual_agg(q.values,pd.to_datetime(q.time.values),kind)
            d['base']=d.cut.map(BASE[kind]); d['delta']=d.value-d.base
            d['pct']=d.delta/d.base*100
            d['variable']=kind; d['model']=m; d['scenario']=ssp
            rows.append(d); del q
            print(f'  {kind} {m} {ssp} ok',flush=True)
TS=pd.concat(rows,ignore_index=True)
TS.to_csv(os.path.join(TB,'C2_hazard_annual_series.csv'),index=False)
print('\nC2', TS.shape, '→ C2_hazard_annual_series.csv')


In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D

TB = rf'{DATA_ROOT}/new_climate/_proj_main/tables'
TS = pd.read_csv(os.path.join(TB,'C2_hazard_annual_series.csv'))

ROLL     = 11
VALUE    = 'pct'                    # 'pct' or 'delta'
SHOW_RAW = True                     # thin lines = annual values, thick lines = moving average
YL   = {'pct':'Change in exceedance days (%)','delta':'Δ days yr$^{-1}$'}[VALUE]
KEEP = {'wave':[1.25,2.5,4.0,6.0],'wind':[8.0,10.8,13.9,17.2]}   # top bin with too thin a baseline is not plotted
SYM  = {'wave':'$H_s$','wind':'$U_{10}$'}
UNIT = {'wave':'m','wind':'m s$^{-1}$'}
VLAB = {'wave':'Significant wave height','wind':'10-m wind speed'}
SC   = {'ssp126':'#2b7bba','ssp585':'#c8452b'}
SLAB = {'ssp126':'SSP1-2.6','ssp585':'SSP5-8.5'}
DASH = {'ACCESS':'-','ECEARTH':(0,(5,2))}
MLAB = {'ACCESS':'ACCESS-CM2','ECEARTH':'EC-Earth3'}

fig, axes = plt.subplots(2, 4, figsize=(16.5, 7.4), dpi=120,
                         sharex=True, sharey=('row' if VALUE=='pct' else False))
for i, kind in enumerate(['wave','wind']):
    for j, c in enumerate(KEEP[kind]):
        ax = axes[i, j]
        d  = TS[(TS.variable==kind) & (TS.cut==c)]
        base = d.base.iloc[0]
        for ssp in ('ssp126','ssp585'):
            for m in ('ACCESS','ECEARTH'):
                s = (d[(d.scenario==ssp)&(d.model==m)]
                     .set_index('year')[VALUE].sort_index())
                if SHOW_RAW:
                    ax.plot(s.index, s.values, ls=DASH[m], color=SC[ssp],
                            lw=.55, alpha=.22, zorder=2)
                ax.plot(s.index, s.rolling(ROLL,center=True,min_periods=ROLL//2+1).mean(),
                        ls=DASH[m], color=SC[ssp], lw=2.1, zorder=4)
        ax.axhline(0, color='#3a3a3a', lw=1, zorder=1)
        ax.set_xlim(2015, 2100); ax.set_xticks(np.arange(2020,2101,20))
        ax.grid(alpha=.2, lw=.5)
        ax.set_title(f'{SYM[kind]} ≥ {c:g} {UNIT[kind]}', fontsize=12, pad=7)
        ax.text(.035,.05, f'base {base:.0f} d/yr', transform=ax.transAxes,
                fontsize=8, color='#7a838a')
        if j == 0:
            ax.set_ylabel(f'{VLAB[kind]}\n{YL}', fontsize=10.5)
        if i == 1:
            ax.set_xlabel('Year', fontsize=11)
        for sp in ('top','right'): ax.spines[sp].set_visible(False)

hd = ([Line2D([],[],color=SC[s],lw=2.2,label=SLAB[s]) for s in SC] +
      [Line2D([],[],color='#1a1a1a',ls=DASH[m],lw=1.9,label=MLAB[m]) for m in MLAB])
fig.legend(handles=hd, loc='upper center', ncol=4, frameon=False, fontsize=12,
           bbox_to_anchor=(.5, 1.005))
fig.suptitle('Exceedance days relative to ERA5 2002–2020 — five regions combined (traffic-weighted)'
             f'   ·   thick lines = {ROLL}-year running mean',
             y=1.06, fontsize=13.5)
plt.tight_layout(); plt.show()


In [ ]:
import os, numpy as np, pandas as pd, xarray as xr

O   = rf'{DATA_ROOT}/new_climate/_proj_main'
TB  = os.path.join(O,'tables')
TA  = pd.read_parquet(os.path.join(TB,'tableA_hazard_monthly.parquet'))

REG = {'NW_Europe':[(36,45,-15,-6),(45,63,-15,-6),(43,63,-6,0),(48,63,0,13)],
       'NW_Atlantic':[(18,31,-98,-60),(31,42,-82,-60),(42,50,-67,-60),(45,55,-60,-46)],
       'Med_Black':[(30,43,-6,0),(30,46,0,19),(30,46,19,26.5),(30,41,26.5,37),(41,47.5,26.5,42)],
       'Asia':[(-10,20,95,130),(20,46,117,146)],
       'Indian_Ocean_N':[(5,30,50,78),(5,26,78,95)]}

cells = xr.open_dataarray(os.path.join(O,'obs_wave.nc')).cell.values
_t = pd.to_datetime(xr.open_dataarray(os.path.join(O,'cm_wave_ACCESS_his.nc')).time.values)
_k = (_t.year>=2002) & (_t.year<=2014)
GOOD = np.ones(len(cells), bool)
for m in ('ACCESS','ECEARTH'):
    for kd in ('wave','wind'):
        GOOD &= np.isfinite(xr.open_dataarray(os.path.join(O,f'cm_{kd}_{m}_his.nc'))
                            .load().values[_k]).mean(0) >= 0.9
def reg_of(g):
    la,lo = map(float,g.split('_'))
    for r,bs in REG.items():
        for S,N,W,E in bs:
            if S<=la<N and W<=lo<E: return r
    return None
RG = np.array([reg_of(g) for g in cells])

# ★ weights = number of usable cells per region (equivalent to a plain mean over all 1,020 cells, no traffic component)
WR = pd.Series({r: float(((RG==r)&GOOD).sum()) for r in REG}); WR = (WR/WR.sum()).rename('w')
print('region weights (pooled cells):'); print(WR.round(3).to_string())

SEASONS = ['DJF','MAM','JJA','SON']
SMAP = {12:'DJF',1:'DJF',2:'DJF',3:'MAM',4:'MAM',5:'MAM',
        6:'JJA',7:'JJA',8:'JJA',9:'SON',10:'SON',11:'SON'}
MIN_BASE_SEA = 0.75          # seasonal baseline <0.75 days → no percentage reported

s = (TA.assign(season=TA.cmonth.map(SMAP))
       .groupby(['model','scenario','slice','region','variable','metric','season'],
                as_index=False)[['base','future']].sum())

# By region
SEA_REG = s.assign(delta=lambda x: x.future-x.base,
                   pct=lambda x: np.where(x.base>=MIN_BASE_SEA,
                                          (x.future-x.base)/x.base*100, np.nan))
# Aggregate
a = (s.merge(WR, left_on='region', right_index=True)
       .assign(b=lambda x: x.base*x.w, f=lambda x: x.future*x.w)
       .groupby(['model','scenario','slice','variable','metric','season'],
                as_index=False)[['b','f','w']].sum())
a['base']=a.b/a.w; a['future']=a.f/a.w
a['delta']=a.future-a.base
a['pct']=np.where(a.base>=MIN_BASE_SEA, a.delta/a.base*100, np.nan)
SEA_AGG = a.drop(columns=['b','f','w'])

SEA_AGG.to_csv(os.path.join(TB,'C3_seasonal_aggregate_cellpooled.csv'), index=False)
SEA_REG.to_csv(os.path.join(TB,'C3_seasonal_region.csv'), index=False)
print('\nSEA_AGG', SEA_AGG.shape, ' SEA_REG', SEA_REG.shape)

# Preview: SSP5-8.5 end of century
q = SEA_AGG[(SEA_AGG.scenario=='ssp585') & (SEA_AGG['slice']=='2081-2100')
            & SEA_AGG.metric.isin(['N_2.5','N_4','N_13.9','N_17.2'])]
print('\n=== five-region aggregate · SSP5-8.5 2081-2100 · % change ===')
print(q.pivot_table(index=['variable','metric'], columns=['season','model'],
                    values='pct').round(1).to_string())


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
plt.rcParams.update({'font.sans-serif':['Arial', 'DejaVu Sans'],
                     'axes.unicode_minus':False})

SLICES = ['2021-2040','2041-2060','2061-2080','2081-2100']
SLAB = {'ssp126':'SSP1-2.6','ssp585':'SSP5-8.5'}
MLAB = {'ACCESS':'ACCESS-CM2','ECEARTH':'EC-Earth3'}
SYM  = {'wave':'$H_s$','wind':'$U_{10}$'}
UNIT = {'wave':'m','wind':'m s$^{-1}$'}
CM   = plt.cm.YlOrRd(np.linspace(.30,.90,4))

def season_bars(variable, cut, value='pct', region=None):
    src = SEA_AGG if region is None else SEA_REG[SEA_REG.region==region]
    d   = src[(src.variable==variable) & (src.metric==f'N_{cut:g}')]
    ylab = {'pct':'Change in exceedance days (%)','delta':'Δ days per season'}[value]
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.2), dpi=120, sharex=True, sharey=True)
    x, w = np.arange(4), .19
    for i, m in enumerate(MLAB):
        for j, ssp in enumerate(SLAB):
            ax  = axes[i, j]
            sub = d[(d.model==m) & (d.scenario==ssp)]
            for k, sl in enumerate(SLICES):
                v = sub[sub['slice']==sl].set_index('season')[value].reindex(SEASONS).values
                ax.bar(x + (k-1.5)*w, v, w*.9, color=CM[k], zorder=3,
                       edgecolor='white', linewidth=.6)
                if k == len(SLICES)-1:                       # label end period only, to avoid clutter
                    for xi, vi in zip(x + (k-1.5)*w, v):
                        if np.isfinite(vi):
                            ax.annotate(f'{vi:.0f}', (xi, vi), ha='center',
                                        va='top' if vi<0 else 'bottom',
                                        xytext=(0, -11 if vi<0 else 3),
                                        textcoords='offset points',
                                        fontsize=7.5, color='#4a4a4a')
            ax.axhline(0, color='#3a3a3a', lw=1, zorder=4)
            ax.set_xticks(x); ax.set_xticklabels(SEASONS, fontsize=11.5)
            ax.grid(axis='y', alpha=.2, lw=.5, zorder=0)
            if i == 0: ax.set_title(SLAB[ssp], fontsize=13, pad=8)
            if j == 0: ax.set_ylabel(f'{MLAB[m]}\n{ylab}', fontsize=10.5)
            for sp in ('top','right'): ax.spines[sp].set_visible(False)
            b = sub[sub['slice']==SLICES[-1]].set_index('season').base.reindex(SEASONS)
            if j == 1:
                ax.text(.985,.04,'baseline  '+' / '.join(f'{v:.0f}' for v in b),
                        transform=ax.transAxes, ha='right', fontsize=7.5, color='#7a838a')
    fig.legend(handles=[Patch(fc=CM[k], label=sl) for k,sl in enumerate(SLICES)],
               loc='upper center', ncol=4, frameon=False, fontsize=11.5, bbox_to_anchor=(.5,1.0))
    ttl = 'five regions pooled (all 2° cells)' if region is None else region
    fig.suptitle(f'Seasonal change in days with {SYM[variable]} ≥ {cut:g} {UNIT[variable]} — {ttl}\n'
                 'relative to ERA5 2002–2020', y=1.075, fontsize=13)
    plt.tight_layout(); plt.show()

season_bars('wave', 4.0)
season_bars('wind', 13.9)


In [ ]:
def season_heat(scenario='ssp585', slice_='2081-2100', value='pct'):
    KEEP = {'wave':[1.25,2.5,4.0,6.0],'wind':[8.0,10.8,13.9,17.2]}
    fig, axes = plt.subplots(2, 2, figsize=(9.5, 6.6), dpi=120)
    d0 = SEA_AGG[(SEA_AGG.scenario==scenario) & (SEA_AGG['slice']==slice_)]
    v  = np.nanpercentile(np.abs(d0[value].dropna()), 96)
    for i, kind in enumerate(['wave','wind']):
        for j, m in enumerate(MLAB):
            ax = axes[i, j]
            M = (d0[(d0.variable==kind)&(d0.model==m)
                    &(d0.metric.isin([f'N_{c:g}' for c in KEEP[kind]]))]
                 .pivot_table(index='metric', columns='season', values=value)
                 .reindex(index=[f'N_{c:g}' for c in KEEP[kind]], columns=SEASONS))
            im = ax.imshow(M.values, cmap='RdYlBu', vmin=-v, vmax=v, aspect='auto')
            for a_ in range(M.shape[0]):
                for b_ in range(M.shape[1]):
                    z = M.values[a_, b_]
                    if np.isfinite(z):
                        ax.text(b_, a_, f'{z:.0f}', ha='center', va='center', fontsize=9,
                                color='white' if abs(z) > .62*v else '#16202a')
            ax.set_xticks(range(4)); ax.set_xticklabels(SEASONS, fontsize=10)
            ax.set_yticks(range(len(KEEP[kind])))
            ax.set_yticklabels([f'≥{c:g} {UNIT[kind]}' for c in KEEP[kind]]
                               if j==0 else [], fontsize=9.5)
            ax.set_title(f'{MLAB[m]} · {SYM[kind]}', fontsize=11.5, pad=6)
            ax.tick_params(length=0)
    cb = fig.colorbar(im, ax=axes, shrink=.6, pad=.02)
    cb.set_label({'pct':'Change (%)','delta':'Δ days per season'}[value])
    fig.suptitle(f'{SLAB[scenario]} · {slice_} · seasonal change by threshold '
                 '(five regions pooled)', y=.99, fontsize=13)
    plt.show()

season_heat('ssp585','2081-2100')
season_heat('ssp126','2081-2100')
